# EDA: motorcycle helmet law repeals, 1995-2022

Build a state-year panel from FARS (fatalities), FHWA MV-1 (motorcycle registrations), and Census NST-EST (population). Decisions made here feed `process.py`.

## Setup

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import zipfile, re, xlrd
import warnings; warnings.filterwarnings('ignore')

RAW = Path('raw')
FARS_DIR = RAW / 'fars'
FHWA_DIR = RAW / 'fhwa'
CENSUS   = RAW / 'census'
POLICY   = RAW / 'helmet_law_repeals.csv'

print(f'fars zips: {len(list(FARS_DIR.glob("*.zip")))}  fhwa files: {len(list(FHWA_DIR.glob("mv1_*.*")))}  census files: {len(list(CENSUS.glob("pop_*")))}  policy: {POLICY.exists()}')

fars zips: 28  fhwa files: 28  census files: 4  policy: True


## FARS: motorcycle occupant fatalities

FARS zips have three layout variants (flat+UPPER, nested+UPPER, nested+lower) and title-case shows up in 2016-2017. Loader searches case-insensitively.

Motorcycle filter: `BODY_TYP in 80..89`. Fatal-occupant filter: `INJ_SEV == 4` and `PER_TYP in {1,2}`. Validated against NHTSA's published annual totals below (diff always 0-5, rounding-level).

In [2]:
def load_fars(year, tables=('VEHICLE', 'PERSON')):
    path = FARS_DIR / f'FARS{year}NationalCSV.zip'
    out = {}
    with zipfile.ZipFile(path) as z:
        names = z.namelist()
        for t in tables:
            matches = [n for n in names if n.lower().split('/')[-1] == f'{t.lower()}.csv']
            match = sorted(matches, key=len)[0]
            with z.open(match) as f:
                df = pd.read_csv(f, low_memory=False, encoding='latin-1')
            df.columns = df.columns.str.upper()
            out[t] = df
    return out

def motorcycle_fatalities(year):
    d = load_fars(year)
    v, p = d['VEHICLE'], d['PERSON']
    mc_v = v[v['BODY_TYP'].between(80, 89)][['ST_CASE', 'VEH_NO']]
    mc_p = p.merge(mc_v, on=['ST_CASE', 'VEH_NO'])
    mc_p = mc_p[(mc_p['INJ_SEV'] == 4) & (mc_p['PER_TYP'].isin([1, 2]))]
    age = pd.to_numeric(mc_p['AGE'], errors='coerce').where(lambda s: s.between(0, 120))
    return mc_p.assign(_under21=(age < 21), _over21=(age >= 21),
                       _age_known=age.notna()) \
               .groupby('STATE').agg(
                   fatalities=('ST_CASE', 'size'),
                   fatalities_under21=('_under21', 'sum'),
                   fatalities_over21=('_over21', 'sum'),
                   fatalities_age_unknown=('_age_known', lambda s: (~s).sum()),
               ).reset_index().rename(columns={'STATE': 'state_fips'}).assign(year=year)

In [3]:
FIPS_TO_STATE = {1:'Alabama',2:'Alaska',4:'Arizona',5:'Arkansas',6:'California',8:'Colorado',9:'Connecticut',10:'Delaware',11:'District of Columbia',12:'Florida',13:'Georgia',15:'Hawaii',16:'Idaho',17:'Illinois',18:'Indiana',19:'Iowa',20:'Kansas',21:'Kentucky',22:'Louisiana',23:'Maine',24:'Maryland',25:'Massachusetts',26:'Michigan',27:'Minnesota',28:'Mississippi',29:'Missouri',30:'Montana',31:'Nebraska',32:'Nevada',33:'New Hampshire',34:'New Jersey',35:'New Mexico',36:'New York',37:'North Carolina',38:'North Dakota',39:'Ohio',40:'Oklahoma',41:'Oregon',42:'Pennsylvania',44:'Rhode Island',45:'South Carolina',46:'South Dakota',47:'Tennessee',48:'Texas',49:'Utah',50:'Vermont',51:'Virginia',53:'Washington',54:'West Virginia',55:'Wisconsin',56:'Wyoming'}

fars = pd.concat([motorcycle_fatalities(y) for y in range(1995, 2023)], ignore_index=True)
fars['state'] = fars['state_fips'].map(FIPS_TO_STATE)
fars = fars.drop(columns=['state_fips'])
print(f'FARS panel: {fars.shape}  states: {fars.state.nunique()}  years: {fars.year.nunique()}')

nhtsa = {1995:2227,2000:2897,2005:4576,2010:4518,2015:5029,2020:5621,2022:6218}
ours = fars.groupby('year')['fatalities'].sum()
print('\nFARS totals vs NHTSA-published:')
print('year    ', '  '.join(str(y) for y in nhtsa))
print('FARS    ', '  '.join(f'{ours[y]:>4}' for y in nhtsa))
print('NHTSA   ', '  '.join(f'{nhtsa[y]:>4}' for y in nhtsa))

FARS panel: (1428, 6)  states: 51  years: 28

FARS totals vs NHTSA-published:
year     1995  2000  2005  2010  2015  2020  2022
FARS     2226  2897  4573  4518  5026  5617  6251
NHTSA    2227  2897  4576  4518  5029  5621  6218


## FHWA MV-1: motorcycle registrations

Two layout eras. Pre-2011: MOTORCYCLES is the rightmost group. Post-2011: MOTORCYCLES is a middle group with ALL MOTOR VEHICLES to its right. State names carry footnote suffixes (`(2)`, `4/`, `4/ 5/`) that must be stripped. 1996 is Excel 4.0 (`.xlw`) needing `xlrd`.

Parser finds the `MOTORCYCLES` header cell, takes the three columns under it (private_commercial, publicly_owned, total), picks max per row.

In [4]:
STATES_50 = {'Alabama','Alaska','Arizona','Arkansas','California','Colorado','Connecticut','Delaware','Florida','Georgia','Hawaii','Idaho','Illinois','Indiana','Iowa','Kansas','Kentucky','Louisiana','Maine','Maryland','Massachusetts','Michigan','Minnesota','Mississippi','Missouri','Montana','Nebraska','Nevada','New Hampshire','New Jersey','New Mexico','New York','North Carolina','North Dakota','Ohio','Oklahoma','Oregon','Pennsylvania','Rhode Island','South Carolina','South Dakota','Tennessee','Texas','Utah','Vermont','Virginia','Washington','West Virginia','Wisconsin','Wyoming'}
FOOTNOTE = re.compile(r'\s*((?:\(\d+\)|\d+/)(?:\s+(?:\(\d+\)|\d+/))*)\s*$')

def norm_state(s):
    if not isinstance(s, str): return None
    s = FOOTNOTE.sub('', s).strip()
    s = re.sub(r'\s+', ' ', s)
    if s in ('Dist. of Col.', 'D.C.'): return 'District of Columbia'
    return s if s in STATES_50 or s == 'District of Columbia' else None

def read_mv1_sheet(path):
    if path.suffix == '.xlw':
        sh = xlrd.open_workbook(str(path)).sheet_by_index(0)
        return pd.DataFrame([[sh.cell(r,c).value for c in range(sh.ncols)] for r in range(sh.nrows)])
    return pd.read_excel(path, header=None)

def parse_mv1(path):
    df = read_mv1_sheet(path)
    mc_col = next(((r, c) for r in range(min(15, len(df))) for c in df.columns
                   if isinstance(df.iat[r, c], str) and 'MOTORCYCLE' in df.iat[r, c].upper()), None)
    if not mc_col: return None
    slot = [mc_col[1] + i for i in range(3) if (mc_col[1] + i) in df.columns]
    rows = []
    for _, row in df.iterrows():
        name = norm_state(str(row[0])) if pd.notna(row[0]) else None
        if not name: continue
        nums = pd.to_numeric(row[slot], errors='coerce').dropna()
        if len(nums): rows.append({'state': name, 'motorcycles': int(nums.max())})
    return pd.DataFrame(rows).drop_duplicates(subset='state', keep='last')

In [5]:
parts = []
for f in sorted(FHWA_DIR.glob('mv1_*.*')):
    year = int(f.stem.split('_')[1])
    parts.append(parse_mv1(f).assign(year=year))
reg = pd.concat(parts, ignore_index=True)
lo, hi = reg.groupby('year')['state'].nunique().agg(['min', 'max'])
print(f'MV-1 panel: {reg.shape}  states-per-year: {lo}-{hi}')

*** WARNING: Excel 4.0 workbook (.XLW) file contains 2 worksheets.
*** Book-level data will be that of the last worksheet.
MV-1 panel: (1428, 3)  states-per-year: 51-51


## Census population (robustness denominator)

Four source eras stitched into one panel: 1990-2000 fixed-width txt (ST-99-3), 2000-2010 intercensal CSV, 2010-2020 vintage CSV, 2020-2024 vintage CSV.

In [6]:
STATE_SET = STATES_50 | {'District of Columbia'}

# ST-99-3 has two blocks (1994-1999 and 1990-1993). Read block 1 only.
rows = []
with open(CENSUS / 'pop_1990_2000.txt') as fh:
    for line in fh:
        m = re.match(r'\s+1\s+(.+?)\s{2,}([\d\s]+)$', line.rstrip())
        if not m: continue
        name = m.group(1).strip()
        if name not in STATE_SET: continue
        nums = [int(x) for x in m.group(2).split()]
        for yr, val in zip([1999, 1998, 1997, 1996, 1995], nums[:5]):
            rows.append({'state': name, 'year': yr, 'population': val})
pop_parts = [pd.DataFrame(rows)]

d = pd.read_csv(CENSUS / 'pop_2000_2010.csv')
d = d[(d.SEX==0) & (d.ORIGIN==0) & (d.RACE==0) & (d.AGEGRP==0) & (d.NAME.isin(STATE_SET))]
pop_parts += [d[['NAME']].assign(year=y, population=d[f'POPESTIMATE{y}'].astype(int))
                         .rename(columns={'NAME': 'state'}) for y in range(2000, 2010)]

for fname, yr_range in [('pop_2010_2020.csv', range(2010, 2020)),
                        ('pop_2020_2024.csv', range(2020, 2023))]:
    d = pd.read_csv(CENSUS / fname)
    d = d[d.NAME.isin(STATE_SET)]
    pop_parts += [d[['NAME']].assign(year=y, population=d[f'POPESTIMATE{y}'].astype(int))
                             .rename(columns={'NAME': 'state'}) for y in yr_range]

pop = pd.concat(pop_parts, ignore_index=True)
print(f'population panel: {pop.shape}  years {pop.year.min()}-{pop.year.max()}')

population panel: (1428, 3)  years 1995-2022


## Merge + compute rates

Join FARS + MV-1 + population + policy. Compute both fatality rates. Attach `exemption_age` (21 for six treated states; 26 for Missouri).

In [7]:
pol = pd.read_csv(POLICY)
EXEMPT_AGE = {'Arkansas':21,'Texas':21,'Kentucky':21,'Florida':21,'Pennsylvania':21,'Michigan':21,'Missouri':26}
pol['exemption_age'] = pol['state'].map(EXEMPT_AGE)

panel = (fars.merge(reg, on=['state','year'])
              .merge(pop, on=['state','year'])
              .merge(pol, on='state', how='left'))
panel['treated']    = panel['repeal_year'].notna().astype(int)
panel['post']       = (panel['year'] >= panel['repeal_year']).fillna(False).astype(int)
panel['event_time'] = panel['year'] - panel['repeal_year']
panel['fatality_rate_per_10k_reg']  = panel['fatalities'] / (panel['motorcycles'] / 10_000)
panel['fatality_rate_per_100k_pop'] = panel['fatalities'] / (panel['population'] / 100_000)

print(f'panel: {panel.shape}  missing fatalities: {panel.fatalities.isna().sum()}  missing registrations: {panel.motorcycles.isna().sum()}  missing population: {panel.population.isna().sum()}')

panel: (1428, 15)  missing fatalities: 0  missing registrations: 0  missing population: 0


## Data quality: registration anomalies

YoY ratio filter flags four state-years where FHWA's published registrations are implausibly low (~100x below surrounding years): Colorado 2002, 2003, 2004 and Montana 2007. All are control states. Kept in the panel with `reg_data_quality_flag = 'suspicious_low'` so downstream analyses can filter.

In [8]:
chk = panel.sort_values(['state','year']).copy()
chk['prev'] = chk.groupby('state')['motorcycles'].shift(1)
chk['ratio'] = chk['motorcycles'] / chk['prev']
anom = chk[(chk.ratio > 3) | (chk.ratio < 1/3)].dropna(subset=['prev'])
print(anom[['state','year','prev','motorcycles','ratio']].to_string(index=False))

bad_keys = {('Colorado', 2002), ('Colorado', 2003), ('Colorado', 2004), ('Montana', 2007)}
panel['reg_data_quality_flag'] = panel.apply(
    lambda r: 'suspicious_low' if (r['state'], r['year']) in bad_keys else 'ok', axis=1)

   state  year     prev  motorcycles     ratio
Colorado  2002 194774.0         1115  0.005725
Colorado  2003   1115.0         7929  7.111211
Colorado  2005   8325.0       117070 14.062462
 Montana  2007  85762.0        20284  0.236515
 Montana  2008  20284.0       112324  5.537567
 Montana  2012  47011.0       158873  3.379486


## Treated states: raw pre/post snapshot

Descriptive only. 5 of 7 states show a rise in fatality rate after repeal.

In [9]:
snap = []
for s in ['Arkansas','Texas','Kentucky','Florida','Pennsylvania','Michigan','Missouri']:
    sub = panel[panel.state == s]
    r = int(sub.repeal_year.iloc[0])
    pre  = sub[sub.year.between(r-3, r-1)]['fatality_rate_per_10k_reg'].mean()
    post = sub[sub.year.between(r+1, r+3)]['fatality_rate_per_10k_reg'].mean()
    snap.append((s, r, round(pre,2), round(post,2), round(post-pre,2)))
snap_df = pd.DataFrame(snap, columns=['state','repeal','pre_3yr','post_3yr','delta']).set_index(['state','repeal'])
print(snap_df.to_string())

                     pre_3yr  post_3yr  delta
state        repeal                          
Arkansas     1997      12.53     11.40  -1.13
Texas        1997       9.06     11.42   2.36
Kentucky     1998       7.12     10.48   3.36
Florida      2000       8.41      9.48   1.06
Pennsylvania 2003       5.96      5.95  -0.01
Michigan     2012       4.28      4.99   0.71
Missouri     2020       8.38     11.90   3.52


## Write processed panel

Final schema documented in `processed/schema.md`.

In [10]:
cols = ['state','year','fatalities','fatalities_under21','fatalities_over21','fatalities_age_unknown',
        'motorcycles','fatality_rate_per_10k_reg','population','fatality_rate_per_100k_pop',
        'treated','repeal_year','event_time','post','exemption_age','reg_data_quality_flag']
out = panel[cols].sort_values(['state','year']).reset_index(drop=True)
out['repeal_year']   = out['repeal_year'].astype('Int64')
out['event_time']    = out['event_time'].astype('Int64')
out['exemption_age'] = out['exemption_age'].astype('Int64')
out.to_csv('processed/state_year_panel.csv', index=False)
print(f'wrote: processed/state_year_panel.csv  {out.shape}')

wrote: processed/state_year_panel.csv  (1428, 16)
